## 1. 필요한 모듈 설치

In [2]:
!pip install selenium

## 2. 필요한 모듈 import

In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

# 크롬 브라우저 객체 생성
driver = webdriver.Chrome()

# 웹 페이지 접속
driver.get("https://www.naver.com")

# 파이썬 인터프리터를 3초 대기
time.sleep(3)

# 검색 입력창 찾기
search_box = driver.find_element(By.ID, "query")
print(type(search_box), search_box)

# 검색어 입력 후 엔터
search_box.send_keys("내일 날씨")
search_box.send_keys(Keys.ENTER)

# driver.quit() # 브라우저 닫기 (항상 수행 완료 후 필수)




<class 'selenium.webdriver.remote.webelement.WebElement'> <selenium.webdriver.remote.webelement.WebElement (session="42a6f86e253b9b5fc274a852dc082a27", element="f.7FA18AE80FF0E6D16B1A3B457C7342C6.d.9D754596B55D8D5D2506B86229ECAE23.e.4")>


## WebDriverWait를 이용한 대기

In [7]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10) #  지정된 요소가 보일 때 까지 10초까지 대기

driver.get("https://www.naver.com")

# 화면에 "query" id를 가지는 요소가 나타날때 까지 대기(최대 10초)
wait.until(EC.presence_of_element_located((By.ID, "query")))

# 검색 입력창 찾기
search_box = driver.find_element(By.ID, "query")
print(type(search_box), search_box)

# 검색어 입력 후 엔터
search_box.send_keys("내일 날씨")
search_box.send_keys(Keys.ENTER)


<class 'selenium.webdriver.remote.webelement.WebElement'> <selenium.webdriver.remote.webelement.WebElement (session="135af5991b21b3ecf246e84e73b4b330", element="f.793257ABC6C7EF4A55D4D9F85C93A2B8.d.FD767F5792122B47A8E44EF495F92398.e.4")>


# 인공지능 검색 후 뉴스탭 이동 -> 아래로 스크롤 하기

In [17]:
from selenium.common.exceptions import NoSuchElementException, TimeoutException

query = '인공지능'
driver = webdriver.Chrome()

try:
    wait = WebDriverWait(driver, 10)
    driver.get('https://www.naver.com')

    search_box = wait.until(EC.presence_of_element_located((By.ID, 'query')))
    search_box.clear()
    search_box.send_keys(query)
    search_box.send_keys(Keys.ENTER)

    # 뉴스 탭 버튼이 클릭이 가능한 상태가 될 때 까지 대기(최대 10초)
    news_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, '//*[@id="lnb"]/div[1]/div/div[1]/div/div[1]/div[2]/a'))
    )
    news_button.click()

    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'body')))
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    wait.until(lambda current_driver: current_driver.execute_script('return window.scrollY') > 0)

    wait.until(EC.url_contains('where=news'))
    print('뉴스 탭 클릭 완료')
    print(driver.current_url)
except TimeoutException:
    print('뉴스 탭을 찾지 못했습니다. 네이버 검색 결과 화면 구조를 확인하세요.')
finally:
    #driver.quit()
    print('Chrome 브라우저 종료 완료')

뉴스 탭 클릭 완료
https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5
Chrome 브라우저 종료 완료


## 뉴스 페이지를 목록을 동적으로 스크래핑

In [18]:
query = '인공지능'
driver = webdriver.Chrome()

try:
    wait = WebDriverWait(driver, 10)
    driver.get(f'https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query={query}')

    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'body')))

    news_list_tags = driver.find_elements(
        By.CSS_SELECTOR,
        'div.sds-comps-vertical-layout > div.sds-comps-vertical-layout.sds-comps-full-layout'
    )

    if not news_list_tags:
        print('뉴스 목록을 찾지 못했습니다. 네이버 화면 구조 또는 CSS 선택자를 확인하세요.')

    news_items = []

    for news_tag in news_list_tags:
        try:
            title_tag = news_tag.find_element(
                By.CSS_SELECTOR,
                'div.sds-comps-vertical-layout.sds-comps-full-layout > a'
            )
            description_candidates = news_tag.find_elements(
                By.CSS_SELECTOR,
                'div.sds-comps-vertical-layout.sds-comps-full-layout > a'
            )

            title = title_tag.text.strip()
            link = title_tag.get_attribute('href')
            description = description_candidates[-1].text.strip() if len(description_candidates) > 1 else ''

            if title and link:
                news_items.append({
                    'title': title,
                    'description': description,
                    'link': link,
                })
        except NoSuchElementException:
            # 광고, 추천 영역 등 뉴스 형식과 다른 박스가 섞여 있으면 건너뛴다.
            continue

    print(f'수집된 뉴스 개수: {len(news_items)}')

    for i, item in enumerate(news_items, start=1):
        print(i, item['title'], item['description'], item['link'])
except TimeoutException:
    print('뉴스 검색 결과 페이지 로딩 시간이 초과되었습니다.')
finally:
    #driver.quit()
    print('Chrome 브라우저 종료 완료')


수집된 뉴스 개수: 11
1 KAIST, 인공지능 최대 난제 '발열' 잡을 냉각기술 확보 의료 인공지능(AI) 기업 뷰노가 대한결핵 및 호흡기학회(이사장 유광하)와 호흡기 질환 분야에서 의료 AI 기술의 임상적 활용 및 학술적 발전을 위한 업무협약(MOU)을 체결했다고 16일 밝혔다. 지난 15일 건국대병원 대회의실에서 진행된 협약식에는 유광하 건국대병원 이사장, 최천웅 강동경희대병원... https://www.newsis.com/view/NISX20260616_0003670583
2 KAIST, 인공지능 최대 난제 '발열' 잡을 냉각기술 확보 인공지능(A)I 데이터센터의 냉각전력을 90%나 줄 일 수 있는 발열관리기술이 나왔다. 한국과학기술원(카이스트·KAIST)는 기계공학과 김성진 교수팀과 AX학과 이익진 교수팀이 공동연구를 통해 반도체 칩 내부에 머리카락보다 가는 물길을 새겨 넣는 초고효율 액체 냉각기술을 개발해 AI 반도체의 최대... https://www.newsis.com/view/NISX20260616_0003670583
3 SK바이오팜, 바이오 USA서 인공지능(AI) 신약개발 전략 공개 SK바이오팜은 이번 행사에서 다수의 글로벌 제약사 및 투자자들과의 파트너링 미팅을 통해 신규 협력 기회를 모색하고, 부스를 통해 연구개발과 회사 운영 전반에 걸친 인공지능(AI) 활용 방향을 소개할 예정이다. BIO USA는 美 바이오협회(BIO)가 주관하는 세계 최대 규모의 바이오 산업 행사다. 올해... https://www.hankyung.com/article/202606168441i
4 “업무 방식 혁신 나서야” 롯데 신동빈 회장, 인공지능 전환 속도 롯데가 신동빈 롯데그룹 회장을 필두로 전사적인 인공지능 전환(AX, AI Transformation)에 속도를 낸다고 16일 밝혔다. 롯데에 따르면 신 회장은 이달 5~6일 ‘CEO AI 아카데미’에 참여해 바이브 코딩(Vibe coding, 자연어로 요구 사항을 입력하면 AI가 코드를 구현하는 방식)을 기

## 백그라운드로 스크래핑

In [19]:
query = '인공지능'

options = webdriver.ChromeOptions() # 옵션 객체
options.add_argument("--headless=new") # 백그라운드에서 동작
options.add_argument("--disable-gpu") # gpu 가속 끄기
options.add_argument("--window-size=1920,1080") # 브라우저 너비

driver = webdriver.Chrome(options=options)

try:
    wait = WebDriverWait(driver, 10)
    driver.get(f'https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query={query}')

    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'body')))

    news_list_tags = driver.find_elements(
        By.CSS_SELECTOR,
        'div.sds-comps-vertical-layout > div.sds-comps-vertical-layout.sds-comps-full-layout'
    )

    if not news_list_tags:
        print('뉴스 목록을 찾지 못했습니다. 네이버 화면 구조 또는 CSS 선택자를 확인하세요.')

    news_items = []

    for news_tag in news_list_tags:
        try:
            title_tag = news_tag.find_element(
                By.CSS_SELECTOR,
                'div.sds-comps-vertical-layout.sds-comps-full-layout > a'
            )
            description_candidates = news_tag.find_elements(
                By.CSS_SELECTOR,
                'div.sds-comps-vertical-layout.sds-comps-full-layout > a'
            )

            title = title_tag.text.strip()
            link = title_tag.get_attribute('href')
            description = description_candidates[-1].text.strip() if len(description_candidates) > 1 else ''

            if title and link:
                news_items.append({
                    'title': title,
                    'description': description,
                    'link': link,
                })
        except NoSuchElementException:
            # 광고, 추천 영역 등 뉴스 형식과 다른 박스가 섞여 있으면 건너뛴다.
            continue

    print(f'수집된 뉴스 개수: {len(news_items)}')

    for i, item in enumerate(news_items, start=1):
        print(i, item['title'], item['description'], item['link'])
except TimeoutException:
    print('뉴스 검색 결과 페이지 로딩 시간이 초과되었습니다.')
finally:
    #driver.quit()
    print('Chrome 브라우저 종료 완료')


수집된 뉴스 개수: 11
1 KAIST, 인공지능 최대 난제 '발열' 잡을 냉각기술 확보 의료 인공지능(AI) 기업 뷰노가 대한결핵 및 호흡기학회(이사장 유광하)와 호흡기 질환 분야에서 의료 AI 기술의 임상적 활용 및 학술적 발전을 위한 업무협약(MOU)을 체결했다고 16일 밝혔다. 지난 15일 건국대병원 대회의실에서 진행된 협약식에는 유광하 건국대병원 이사장, 최천웅 강동경희대병원... https://www.newsis.com/view/NISX20260616_0003670583
2 KAIST, 인공지능 최대 난제 '발열' 잡을 냉각기술 확보 인공지능(A)I 데이터센터의 냉각전력을 90%나 줄 일 수 있는 발열관리기술이 나왔다. 한국과학기술원(카이스트·KAIST)는 기계공학과 김성진 교수팀과 AX학과 이익진 교수팀이 공동연구를 통해 반도체 칩 내부에 머리카락보다 가는 물길을 새겨 넣는 초고효율 액체 냉각기술을 개발해 AI 반도체의 최대... https://www.newsis.com/view/NISX20260616_0003670583
3 SK바이오팜, 바이오 USA서 인공지능(AI) 신약개발 전략 공개 SK바이오팜은 이번 행사에서 다수의 글로벌 제약사 및 투자자들과의 파트너링 미팅을 통해 신규 협력 기회를 모색하고, 부스를 통해 연구개발과 회사 운영 전반에 걸친 인공지능(AI) 활용 방향을 소개할 예정이다. BIO USA는 美 바이오협회(BIO)가 주관하는 세계 최대 규모의 바이오 산업 행사다. 올해... https://www.hankyung.com/article/202606168441i
4 “업무 방식 혁신 나서야” 롯데 신동빈 회장, 인공지능 전환 속도 롯데가 신동빈 롯데그룹 회장을 필두로 전사적인 인공지능 전환(AX, AI Transformation)에 속도를 낸다고 16일 밝혔다. 롯데에 따르면 신 회장은 이달 5~6일 ‘CEO AI 아카데미’에 참여해 바이브 코딩(Vibe coding, 자연어로 요구 사항을 입력하면 AI가 코드를 구현하는 방식)을 기